### Creating 3-year centered zarr files

In [ ]:
time_series_years = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
three_year_centered_data = []
for year in time_series_years:
    year_before = year-1
    year_after = year+1
    zarr_files=[rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year_before}_combined.zarr',rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year}_combined.zarr',rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year_after}_combined.zarr']
    data_3year = xr.open_mfdataset(zarr_files,engine="zarr")
    three_year_centered_data.append(data_3year)

### Saving Lowess smoothing to zarr files

In [ ]:
time_series_values = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
shapefile_geometries = [MAB_south_loc, MAB_north_loc, GB_whole_loc, GOM_west_loc, GOM_east_loc]
region_title = ['MABS','MABN','GB','GOMW','GOME']
for year in time_series_values:
    year_index = time_series_values.index(year)
    dataset = three_year_centered_data[year_index]
    for i in range(5):
        region_name = region_title[i]
        shapefile_geometry = shapefile_geometries[i]
        dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
        dataset.rio.write_crs("epsg:4326", inplace=True)
        clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile.crs, drop=True)
        regional_data = clipped_daily.mean(dim=['lat','lon'])   
        time = regional_data.time.astype('int64')
        median=regional_data.to_dataframe().reset_index()
        median = median["CHL_median"]
        smoothed_CHL = sm.nonparametric.smoothers_lowess.lowess(median,time,frac=0.04,return_sorted=False)
        df_output = pd.DataFrame({
            'time':time,
            'CHL_median':smoothed_CHL
        })
        df_output.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\smoothed_3year_{str(region_name)}_data_{time_series_years[year_index]}.csv',index=False)
        print(f"Successfully saved smoothed_3year_{str(region_name)}_data_{time_series_values[year_index]}.csv")

### Creating bloom mask variable in zarr files

In [ ]:
zarr_data = [MABS,MABN,GB,GOMW,GOME]
csv_data = [bloom_MABS,bloom_MABN,bloom_GB,bloom_GOMW,bloom_GOME]
title = ['MABS','MABN','GB','GOMW','GOME']
zarr_path = [
    r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_smoother_tests.zarr',
    r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_smoother_tests.zarr',
    r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_smoother_tests.zarr',
    r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_smoother_tests.zarr',
    r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_smoother_tests.zarr'
]
for i in range(len(title)):
    zarr = zarr_path[i]
    big_data = zarr_data[i]
    small_data = csv_data[i].copy()
    small_data = small_data.dropna(subset=['Start date','End date'])
    small_data['Start date']=pd.to_datetime(small_data['Start date'])
    small_data['End date']=pd.to_datetime(small_data['End date'])
    bloom_mask = xr.DataArray(
        np.zeros(big_data['time'].shape,dtype=bool),
        coords=[big_data['time']],
        dims=['time']
    )
    for index,row in small_data.iterrows():
        start = row['Start date'].to_numpy()
        end = row['End date'].to_numpy()
        in_window = (big_data['time']>= start) & (big_data['time']<= end)
        bloom_mask = bloom_mask | in_window
    big_data['CHL_bloom_only_nan']=big_data['CHL_median'].where(bloom_mask,np.nan)
    new_variable = big_data[['CHL_bloom_only']]
    new_variable.to_zarr(zarr,mode='a')
    print("Success!")

### Creating DataFrames for each region with bloom metrics

### Creating regional summary dataframes

In [ ]:
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_clipped_thld,MABN_clipped_thld,GB_clipped_thld,GOMW_clipped_thld,GOME_clipped_thld]
clipped_median = [MABS_clipped_med,MABN_clipped_med,GB_clipped_med,GOMW_clipped_med,GOME_clipped_med]
for x in range(5):
    df = summary_bloom_metrics(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_region[x])
    df.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Summary_Stats.csv',mode='w')
    print("Successfully saved file :)")

In [ ]:
shapefile_geo = [MAB_south_loc,MAB_north_loc,GB_whole_loc,GOM_west_loc,GOM_east_loc]
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_clipped_thld,MABN_clipped_thld,GB_clipped_thld,GOMW_clipped_thld,GOME_clipped_thld]
for x in range(5):
    # Year and Events per year
    bloom_events_annual = annual_events(shapefile_geo[x],data_region[x],clipped_thld=clipped[x])

    # Bloom days per year
    start_DOY , end_DOY, bloom_events = rolling_peak_window(shapefile_geo[x],clipped_thld=clipped[x],dataset=data_region[x])
    year = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
    bloom_days_annual = []
    for i in year:
        bloom_days = bloom_days_per_year(data_region[x],i,start_DOY=start_DOY,end_DOY=end_DOY)
        bloom_days_annual.append(bloom_days)

    #Bloom integrated chl-a
    integrated_annual = []
    for i in year:
        int_chl = yearly_integrated_chl(data_region[x],i)
        integrated_annual.append(int_chl)

    # Bloom days per year above the threshold
    bloom_days_above = []
    for i in year:
        bloom_days_thld = bloom_days_above_threshold(clipped[x],data_region[x],i,start_DOY=start_DOY, end_DOY=end_DOY, bloom_events=bloom_events)
        bloom_days_above.append(bloom_days_thld)

    data = {
        'Number of blooms': bloom_events_annual,
        'Total bloom days': bloom_days_annual,
        'Bloom days above threshold': bloom_days_above,
        'Total integrated chl': integrated_annual
    }
    df = pd.DataFrame(data,index=year)
    df.index.name = "Year"
    df.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Summary_Metrics_new.csv')
    print("Successfully saved file :)")

In [ ]:
region_acro = ['MABS','MABN','GB','GOMW','GOME']
shapefile_geo = [MAB_south_loc,MAB_north_loc,GB_whole_loc,GOM_west_loc,GOM_east_loc]
data_region = [MABS,MABN,GB,GOMW,GOME]
bloom_data = [summary_MABS,summary_MABN,summary_GB,summary_GOMW,summary_GOME]
clipped = [MABS_clipped_thld,MABN_clipped_thld,GB_clipped_thld,GOMW_clipped_thld,GOME_clipped_thld]
for x in range(5):
    bloom_dataset = bloom_data[x]
    year = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
    month_options = ['January','February','March','April','May','June','July','August','September','October','November','December']
    for j in range(len(month_options)):
        month_actual = j+1
        monthly_chl = []
        for i in year:
            month_chl = monthly_integrated_chl(data_region[x],i,month_actual)
            monthly_chl.append(month_chl)
        bloom_dataset[f"{month_options[j]} Integrated Chlorophyll"] = monthly_chl
        bloom_dataset.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Summary_Metrics_new.csv',mode='w')